# [8.3] ACDC and Circuit Metrics - Exercises

Build a small circuit-evaluation ladder: patch-recovery scores, ACDC-style pruning, faithfulness, minimality, completeness, random baselines, and held-out template checks.

In [ ]:
import sys
from collections.abc import Mapping
from dataclasses import dataclass
from pathlib import Path

import torch as t

chapter = "chapter8_automated_circuits"
section = "part3_acdc_circuit_metrics"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part3_acdc_circuit_metrics.tests as tests

GT_TIER = "GT-1"
EXERCISE_ID = "8.3.acdc_circuit_metrics"
EXPECTED_RUNTIME = "35-50 minutes for exercises; about 1-2 minutes for CUDA preflight"
REQUIRES_GPU = False

In [ ]:
@dataclass(frozen=True)
class ActivationPatchingSweep:
    patch_scores: t.Tensor
    best_index: int
    best_score: float


@dataclass(frozen=True)
class ACDCPruningReport:
    kept_edges: tuple[str, ...]
    removed_edges: tuple[str, ...]
    threshold: float
    num_kept: int


@dataclass(frozen=True)
class CircuitFaithfulnessReport:
    full_metric: float
    corrupt_metric: float
    circuit_metric: float
    preserved_fraction: float
    passes_faithfulness: bool


@dataclass(frozen=True)
class CircuitMinimalityReport:
    circuit_metric: float
    ablated_metric: float
    metric_damage: float
    passes_minimality: bool


@dataclass(frozen=True)
class CircuitCompletenessReport:
    circuit_metric: float
    expanded_metric: float
    omitted_node_gain: float
    passes_completeness: bool


@dataclass(frozen=True)
class RandomCircuitBaselineReport:
    circuit_metric: float
    random_metric: float
    margin: float
    circuit_beats_random: bool


@dataclass(frozen=True)
class OODTemplateReport:
    per_template_accuracy: dict[int, float]
    worst_template_accuracy: float
    passes_ood: bool

@dataclass(frozen=True)
class CircuitMethodComparisonReport:
    exact_top_edges: tuple[str, ...]
    method_top_edges: dict[str, tuple[str, ...]]
    topk_overlap: dict[str, float]
    score_correlations: dict[str, float]
    circuit_sizes: dict[str, int]
    best_matching_method: str
    passes_comparison: bool

## Patch-Recovery Scores

Convert clean, corrupt, and patched metrics into normalized recovery scores.

In [ ]:
def answer_logit_diff(
    logits: t.Tensor,
    *,
    positive_token_id: int,
    negative_token_id: int,
) -> float:
    raise NotImplementedError()


def activation_patching_sweep(
    *,
    clean_metric: float,
    corrupt_metric: float,
    patched_metrics: t.Tensor,
) -> ActivationPatchingSweep:
    raise NotImplementedError()


tests.test_position_patching_helpers_score_recovery(
    answer_logit_diff,
    activation_patching_sweep,
)

## ACDC-Style Pruning

Keep edge names whose scores survive a threshold.

In [ ]:
def acdc_pruning_report(
    edge_scores: t.Tensor,
    edge_names: list[str],
    *,
    threshold: float,
) -> ACDCPruningReport:
    raise NotImplementedError()


tests.test_acdc_pruning_report_keeps_threshold_edges(acdc_pruning_report)

## Faithfulness

Measure how much clean-vs-corrupt behavior the candidate circuit preserves.

In [ ]:
def circuit_faithfulness_report(
    *,
    full_metric: float,
    corrupt_metric: float,
    circuit_metric: float,
    min_preserved_fraction: float = 0.75,
) -> CircuitFaithfulnessReport:
    raise NotImplementedError()


tests.test_faithfulness_report_normalizes_clean_corrupt_gap(
    circuit_faithfulness_report,
)

## Minimality And Completeness

Keep these as separate reports: one detects bloated circuits, the other detects missing nodes.

In [ ]:
def circuit_minimality_report(
    *,
    circuit_metric: float,
    ablated_metric: float,
    min_metric_damage: float = 0.5,
) -> CircuitMinimalityReport:
    raise NotImplementedError()


def circuit_completeness_report(
    *,
    circuit_metric: float,
    expanded_metric: float,
    max_omitted_node_gain: float = 0.2,
) -> CircuitCompletenessReport:
    raise NotImplementedError()


tests.test_minimality_and_completeness_reports_distinguish_failure_modes(
    circuit_minimality_report,
    circuit_completeness_report,
)

## Random And OOD Controls

Beat a same-size random circuit, and expose worst-template accuracy on held-out prompt families.

In [ ]:
def random_circuit_baseline_report(
    *,
    circuit_metric: float,
    random_metric: float,
    min_margin: float = 0.5,
) -> RandomCircuitBaselineReport:
    raise NotImplementedError()


tests.test_random_circuit_baseline_report_requires_margin(
    random_circuit_baseline_report,
)

In [ ]:
def ood_template_report(
    logits: t.Tensor,
    answer_ids: t.Tensor,
    template_ids: t.Tensor,
    *,
    min_accuracy: float = 0.75,
) -> OODTemplateReport:
    raise NotImplementedError()


tests.test_ood_template_report_tracks_worst_template(ood_template_report)

## Exact-vs-Approximate Circuit Comparison

ACDC-style pruning is only useful if approximate discovery methods recover the same important edges as exact interventions. Compare exact patch scores with approximate scores by top-k overlap, score correlation, and circuit size.

In [ ]:
def circuit_method_comparison_report(
    exact_scores: t.Tensor,
    method_scores: Mapping[str, t.Tensor],
    edge_names: list[str],
    *,
    top_k: int,
    min_topk_overlap: float = 0.5,
    min_score_correlation: float = 0.5,
) -> CircuitMethodComparisonReport:
    raise NotImplementedError()


tests.test_circuit_method_comparison_report_matches_exact_patching(
    circuit_method_comparison_report,
)

## Full Verification Contract

The smoke tests check the local exercise implementation. This final cell checks the committed CUDA verification report for the section-scale run and exposes the same `run_gpu_test` / `run_full_experiment` surface used by the release gate.


In [ ]:
def _load_committed_gpu_report() -> dict:
    import json

    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = _load_committed_gpu_report()
{key: gpu[key] for key in [
    "device",
    "preflight_passed",
    "peak_vram_gb",
] if key in gpu}
